# 07 · Adaptación al cambio nacional y calibración por familia electoral

**Pregunta.** ¿Puede corregirse el fallo de cobertura 2021 usando únicamente señales disponibles antes de cada elección, sin perder robustez en 2024?

**Entradas.** Feature mart A+B, predicciones del candidato aprobado, intervalos del notebook 06 y `configs/calibration_adaptation.yaml`.

**Método.** La predicción suma el cambio nacional observado en el ciclo anterior de la misma familia, separado por seis años. Se comparan el conformal móvil original, el conformal móvil con centro corregido y un intervalo de análogo familiar calibrado con la última elección de la misma familia. Cuando el candidato no tiene predicción fuera de tiempo para el análogo 2015, se usa explícitamente el baseline de mismo tipo como arranque frío.

**Criterios.** Todas las fuentes deben preceder al target; el MAE 2021 debe mejorar; el deterioro 2024 no puede superar 0.5 pp; la cobertura global debe superar 85% en ambos targets. Alcanzar ese piso habilita experimentación 2027, pero sólo 90% en todos los targets permitiría hablar de intervalos publicables.

**Salidas.** Correcciones nacionales, predicciones corregidas, comparación de intervalos, decisión máquina-legible y reporte HTML.

## Configuración y entradas

Se fija la regla antes de evaluar resultados. No se aprende un factor con 2021 ni 2024: se traslada exactamente el cambio nacional del ciclo comparable anterior.

In [ ]:
from pathlib import Path
import json
import time

import pandas as pd
import yaml
from IPython.display import Markdown, display

from radar27.modeling.calibration_adaptation import (
    build_adaptive_interval_comparison,
    build_shift_corrected_predictions,
    estimate_national_shift,
    write_adaptation_decision,
)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (
            candidate / "configs/calibration_adaptation.yaml"
        ).exists():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RUN_STARTED = time.perf_counter()
with (PROJECT_ROOT / "configs/calibration_adaptation.yaml").open(encoding="utf-8") as stream:
    adaptation_config = yaml.safe_load(stream)
required_inputs = [
    PROJECT_ROOT / "artifacts/tables/feature_mart_ab.parquet",
    PROJECT_ROOT / "artifacts/tables/model_backtest_predictions.parquet",
    PROJECT_ROOT / "artifacts/tables/conformal_backtest_intervals.parquet",
    PROJECT_ROOT / "artifacts/uncertainty_decision.json",
]
input_checks = {path.relative_to(PROJECT_ROOT).as_posix(): path.exists() for path in required_inputs}
display(pd.Series(input_checks, name="existe").to_frame())
display(pd.Series(adaptation_config["national_shift"], name="valor").to_frame())
assert all(input_checks.values())

## Corrección nacional fuera de tiempo

Para cada target se calcula la diferencia ponderada `actual - rezago de misma familia` seis años antes. La fila 2027 es sólo un parámetro operacional derivado de 2021; no es una predicción.

In [ ]:
shift_estimates = estimate_national_shift(PROJECT_ROOT)
shift_predictions, shift_metrics = build_shift_corrected_predictions(
    PROJECT_ROOT, shift_estimates
)
display(shift_estimates)
display(shift_metrics)

In [ ]:
shift_checks = {
    "all_sources_strictly_prior": bool(shift_estimates.source_strictly_prior.all()),
    "evaluated_families_match": bool(
        shift_estimates.loc[
            shift_estimates.target_year.isin([2021, 2024, 2027]), "family_match"
        ].all()
    ),
    "probability_bounds": bool(
        shift_predictions.prediction_shift_corrected.dropna().between(0, 1).all()
    ),
    "primary_mae_improves": bool(
        shift_metrics.set_index("target_year").loc[2021, "weighted_mae_change"] < 0
    ),
    "secondary_deterioration_within_05pp": bool(
        shift_metrics.set_index("target_year").loc[2024, "weighted_mae_change"] <= 0.005
    ),
}
display(pd.Series(shift_checks, name="aprobado").to_frame())
assert all(shift_checks.values())

In [ ]:
primary = shift_metrics.set_index("target_year").loc[2021]
display(Markdown(
    f"**Resultado puntual 2021.** El MAE ponderado cambia de "
    f"`{100*primary.original_weighted_mae:.2f}` a `{100*primary.corrected_weighted_mae:.2f}` pp "
    f"y el sesgo de `{100*primary.original_weighted_bias:.2f}` a "
    f"`{100*primary.corrected_weighted_bias:.2f}` pp."
))

## Comparación de intervalos adaptativos

El método de análogo familiar cambia la distribución de calibración, no sólo el centro. Para 2021 usa errores del baseline 2015 porque el candidato aún no tenía historia entrenable; para 2024 usa los errores fuera de tiempo del candidato 2018. Esta adaptación es empírica y no se presenta como garantía conformal bajo intercambiabilidad.

In [ ]:
adaptive_intervals, adaptive_coverage, adaptive_calibration = (
    build_adaptive_interval_comparison(PROJECT_ROOT, shift_predictions)
)
overall_coverage = adaptive_coverage[
    adaptive_coverage.group_dimension.eq("overall")
].copy()
display(overall_coverage[[
    "interval_method", "target_year", "empirical_coverage",
    "identified_weighted_coverage", "mean_interval_width",
]])
display(adaptive_calibration)

In [ ]:
interval_checks = {
    "calibration_strictly_prior": bool(
        (adaptive_intervals.max_calibration_target_year < adaptive_intervals.target_year).all()
    ),
    "one_interval_per_method_unit": not adaptive_intervals.duplicated(
        ["interval_method", "target_year", "evaluation_unit_id"]
    ).any(),
    "ordered_bounds": bool((adaptive_intervals.lower <= adaptive_intervals.upper).all()),
    "bounds_are_probabilities": bool(
        adaptive_intervals.lower.dropna().between(0, 1).all()
        and adaptive_intervals.upper.dropna().between(0, 1).all()
    ),
}
display(pd.Series(interval_checks, name="aprobado").to_frame())
assert all(interval_checks.values())

## Decisión

Se distingue entre habilitar una primera canalización experimental 2027 y afirmar que los intervalos son publicables. El segundo estándar exige cobertura nominal en todos los targets y es deliberadamente más estricto.

In [ ]:
adaptation_decision = write_adaptation_decision(
    PROJECT_ROOT, shift_metrics, adaptive_coverage
)
display(pd.Series(adaptation_decision).to_frame("valor"))

In [ ]:
report_path = PROJECT_ROOT / "artifacts/reports/calibration_adaptation.html"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join([
    "<h1>RADAR 27 · Adaptación de calibración</h1>",
    "<p>Generado por notebooks/07_calibration_adaptation.ipynb.</p>",
    "<h2>Correcciones nacionales</h2>", shift_estimates.to_html(index=False),
    "<h2>Métricas puntuales</h2>", shift_metrics.to_html(index=False),
    "<h2>Comparación de cobertura</h2>", overall_coverage.to_html(index=False),
    "<h2>Calibración</h2>", adaptive_calibration.to_html(index=False),
    "<h2>Decisión</h2><pre>" + json.dumps(
        adaptation_decision, ensure_ascii=False, indent=2
    ) + "</pre>",
]), encoding="utf-8")

required_outputs = [
    PROJECT_ROOT / "artifacts/tables/national_shift_estimates.csv",
    PROJECT_ROOT / "artifacts/tables/shift_corrected_predictions.parquet",
    PROJECT_ROOT / "artifacts/tables/shift_correction_metrics.csv",
    PROJECT_ROOT / "artifacts/tables/adaptive_interval_predictions.parquet",
    PROJECT_ROOT / "artifacts/tables/adaptive_interval_coverage.csv",
    PROJECT_ROOT / "artifacts/tables/adaptive_interval_calibration.csv",
    PROJECT_ROOT / "artifacts/calibration_adaptation_decision.json",
    report_path,
]
final_checks = {
    "all_outputs_exist": all(path.exists() for path in required_outputs),
    **shift_checks,
    **interval_checks,
}
display(pd.Series(final_checks, name="aprobado").to_frame())
assert all(final_checks.values())

In [ ]:
display(Markdown(
    f"**Cierre.** Experimental 2027: "
    f"`{adaptation_decision['approved_for_experimental_2027_pipeline']}`. "
    f"Intervalo publicable: `{adaptation_decision['public_interval_ready']}`. "
    f"Tiempo total: `{time.perf_counter()-RUN_STARTED:.1f}` s."
))